LIMS Django Application 

This notebook contains information pertaining to the creation of the LIMS application using Django. 

OUTLINE: 
Startup - creating the app 
Homepage setup 
Creating Admin Page 
Switching cd 
Creating Tables through models.py for PostgresSQL server in PgAdmin4
Issues with Interpreter 


# Startup 
# macOS 
python 3 -m venv . venv 
source .venv/bin/activate

# Create the virtual environment 
# Open the virtual command palette (View -> Command Palette) and type "Python: Select Interpreter" to select the virtual environment you created.
# Select the interpreter from the virtual environment you created, which should be something like `.venv/bin/python`.

# update pip 
python -m pip install --upgrade pip
# install Django 
python -m pip install django

#Create a new Django project
django-admin startproject appsett 
# appsett is the name of the project I created
#This will create a new project folder with manage.py and other startup subfolders 

# Run the server for the first time 
 python manage.py runserver 
# This will start the development server, and you can access it at http://127.0.1:8000/ and it will show the default Django welcome page.

# Create a new Django app
python manage.py startapp inventory 
# This will create a new app folder with models.py, views.py, and other files.
# In the settings.py file of the project, you need to add the new app to the INSTALLED_APPS list:
# Open appsett/settings.py and add 'inventory' to the INSTALLED_APPS list
INSTALLED_APPS = [
'inventory',  # Add your app here
]



# Adding a view for the home page 
# Open inventory/views.py and add the following code:
from django.http import HttpResponse
def home(request):
return HttpResponse("Welcome to LIMS !")

# Map view to URL 
# Open inventory/urls.py and add the following code:
from django.urls import path
from .views import home
urlpatterns = [
path('', home, name='home'),
]
# Include the app's URLs in the project's URL configuration
# Open appsett/urls.py and modify it as follows:
from django.contrib import admin
from django.urls import path, include
urlpatterns = [
path('admin/', admin.site.urls),
path('', include('inventory.urls')),  # Include the app's URLs
]
# Now, when you run the server again with python manage.py runserver and visit http://127.0.1:8000/, you should see "Welcome to LIMS !".

# Creating Admin Interface (Django Admin) 
# To create an admin interface, you need to create a superuser account.
python manage.py createsuperuser
# Follow the prompts to create a superuser account with a username, email, and password. (Remember these credentials)
# After creating the superuser, you can access the admin interface at http://127.0.1:8000/admin/ and log in with the superuser credentials you just created.
#This allows you to manage the user accounts, groups, and other administrative tasks.

# Remembering how to switch the current working directory 
    cd /Users/gracedurham/Desktop/limsproject/appsett  
# limsproject is the folder that was created on my desktop to hold the project "appsett"


# Creating the tables for PostgreSQL 
# in inventory/models.py, define your models: 
# Example model -
from django.db import models
class Item(models.Model):
    name = models.CharField(max_length=100)
    quantity = models.IntegerField()
    description = models.TextField()

    def __str__(self):
        return self.name
# After defining your models, you need to create the database tables:
python manage.py makemigrations
python manage.py migrate
# This will create the necessary database tables based on your models.

# Verify the database table was created in PgAdmin 
# Open PgAdmin and connect to your PostgreSQL database.
# You should see the new table under the database name (limsapp) -> "Schemas" -> "Tables". If not, right click on "Tables" and refresh the view.


# If imports have an error : 

# In terminal, run: 
where python  
# this will give you the location of the virtual environment 
# On mac -> (Cmd + Shift + P) then type or select 
Python: Select Interpreter
# make sure the end of the interpreter matches the location from "where python" in terminal 

SQL original Query 

SELECT * FROM public.inventory_inventory
ORDER BY id ASC LIMIT 100


SQL Queries to fix the missing rows of data issue - AKA the import skipped over 17 rows of data in the original import. Fixed now using all this code - I know it's a lot, just keeping record of how that all got fixed. not necessarily relevant 

SELECT COUNT(*) FROM inventory_inventory; 

CREATE TABLE temp_import (LIKE inventory_inventory INCLUDING ALL);

DROP TABLE IF EXISTS temp_import;

CREATE TABLE temp_import (
    reorder TEXT,
    item_name TEXT,
    sku TEXT,
    vendor TEXT,
    location TEXT,
    sub_location TEXT,
    location_detail TEXT,
    amount_in_stock TEXT,
    units TEXT,
    vendor_qty TEXT,
    date_opened TEXT,
    expiration_date TEXT,
    storage_temperature TEXT,
    storage_instructions TEXT,
    notes TEXT,
    url TEXT
);



SELECT COUNT(*) FROM temp_import;
SELECT * FROM temp_import ORDER BY item_name LIMIT 20;

SELECT * FROM temp_import 
WHERE sku IS NULL OR item_name IS NULL 
LIMIT 20; 


-- Convert numeric columns
ALTER TABLE temp_import
ALTER COLUMN amount_in_stock TYPE INTEGER USING NULLIF(amount_in_stock,'')::INTEGER,
ALTER COLUMN vendor_qty TYPE INTEGER USING NULLIF(vendor_qty,'')::INTEGER;

-- Convert date columns
UPDATE temp_import SET date_opened = NULL WHERE date_opened !~ '^\d{4}-\d{2}-\d{2}$';
ALTER TABLE temp_import
ALTER COLUMN date_opened TYPE DATE USING date_opened::DATE;

UPDATE temp_import SET expiration_date = NULL WHERE expiration_date !~ '^\d{4}-\d{2}-\d{2}$';
ALTER TABLE temp_import
ALTER COLUMN expiration_date TYPE DATE USING expiration_date::DATE;

ALTER TABLE temp_import RENAME COLUMN "SKU" TO sku;
ALTER TABLE temp_import RENAME COLUMN "URL" TO url;

SELECT column_name 
FROM information_schema.columns 
WHERE table_name = 'temp_import'

SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_name = 'temp_import';

SELECT column_name
FROM information_schema.columns
WHERE table_name = 'temp_import'
ORDER BY ordinal_position;

SELECT * 
FROM inventory_inventory 
LIMIT 5;


ALTER TABLE temp_import RENAME COLUMN sku TO "SKU";
ALTER TABLE temp_import RENAME COLUMN url TO "URL";

INSERT INTO inventory_inventory(
    reorder, item_name, "SKU", vendor, location, sub_location,
    location_detail, amount_in_stock, units, vendor_qty,
    date_opened, expiration_date, storage_temperature,
    storage_instructions, notes, "URL"
)
SELECT reorder, item_name, "SKU", vendor, location, sub_location,
       location_detail, amount_in_stock, units, vendor_qty,
       date_opened, expiration_date, storage_temperature,
       storage_instructions, notes, "URL"
FROM temp_import t
WHERE "SKU" IS NOT NULL
  AND NOT EXISTS (
      SELECT 1
      FROM inventory_inventory i
      WHERE i."SKU" = t."SKU"
  );

UPDATE temp_import
SET units = 'Unknown Units'
WHERE units IS NULL;

SELECT *
FROM temp_import
WHERE "SKU" IS NULL
   OR vendor IS NULL
   OR units IS NULL;





ALTER TABLE inventory_inventory
ALTER COLUMN date_opened DROP NOT NULL;

UPDATE temp_import 
SET "SKU" = TRIM("SKU");

SELECT t."SKU"
FROM temp_import t
LEFT JOIN inventory_inventory i
  ON i."SKU" = t."SKU"
WHERE i."SKU" IS NULL;



SELECT column_name 
FROM information_schema.columns 
WHERE table_name = 'inventory_inventory'


SELECT column_name
FROM information_schema.columns
WHERE table_name = 'inventory_inventory';


SELECT COUNT(*) FROM inventory_inventory;
SELECT COUNT(*) FROM temp_import;
SELECT COUNT(*) FROM temp_import t
WHERE NOT EXISTS (
    SELECT 1 FROM inventory_inventory i
    WHERE i."SKU" = t."SKU"
);


SELECT "SKU", LENGTH("SKU") AS len
FROM temp_import
WHERE "SKU" IS NOT NULL;

WITH null_skus AS (
    SELECT "SKU", item_name, ROW_NUMBER() OVER () AS rn
    FROM temp_import
    WHERE "SKU" IS NULL
)
UPDATE temp_import t
SET "SKU" = 'N/A' || sub.rn
FROM (
    SELECT ctid, ROW_NUMBER() OVER () AS rn
    FROM temp_import
    WHERE "SKU" IS NULL
) AS sub
WHERE t.ctid = sub.ctid;

SELECT "SKU", item_name
FROM temp_import
WHERE "SKU" LIKE 'N/A%';


UPDATE temp_import
SET vendor = 'Unknown Vendor'
WHERE vendor IS NULL;

UPDATE temp_import
SET location = 'Unknown Location'
WHERE location IS NULL;

UPDATE temp_import
SET amount_in_stock = 0
WHERE amount_in_stock IS NULL;

SELECT *
FROM temp_import
WHERE "SKU" IS NULL
   OR vendor IS NULL
   OR units IS NULL
   OR location IS NULL
   OR amount_in_stock IS NULL;




SELECT column_name
FROM information_schema.columns
WHERE table_name = 'inventory_inventory'
  AND is_nullable = 'NO';

SELECT "SKU", item_name, vendor, units, location
FROM inventory_inventory
ORDER BY "SKU" DESC
LIMIT 20;



SELECT column_name
FROM information_schema.columns
WHERE table_name = 'inventory_inventory';

ALTER TABLE inventory_inventory RENAME COLUMN "URL" TO url;
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'inventory_inventory';

SELECT "SKU", item_name, vendor, units, location, url
FROM inventory_inventory
ORDER BY "SKU" DESC
LIMIT 20;

SELECT "SKU", item_name, date_opened
FROM inventory_inventory
WHERE date_opened IS NOT NULL
ORDER BY date_opened DESC
LIMIT 20;

SELECT column_name, column_default
FROM information_schema.columns
WHERE table_name = 'inventory_inventory'
  AND column_name = 'date_opened';

UPDATE inventory_inventory
SET date_opened = NULL
WHERE "SKU" LIKE 'N/A%';

SELECT "SKU", item_name, date_opened
FROM inventory_inventory
WHERE "SKU" LIKE 'N/A%';

SELECT "SKU", item_name, date_opened
FROM inventory_inventory
WHERE date_opened = '2025-09-05';

UPDATE inventory_inventory
SET date_opened = NULL
WHERE date_opened = '2025-09-05'
  AND "SKU" NOT LIKE 'N/A%';  -- keeps the 17 placeholder rows separate if needed

UPDATE inventory_inventory
SET date_opened = NULL
WHERE "SKU" NOT LIKE 'N/A%';  -- only affects original rows, not placeholders

ALTER TABLE inventory_inventory 
DROP COLUMN quantity;


